# RealMLP Approach on Predicting Heart Disease

Purpose: 


### Package import

In [ ]:
# !pip install pytabkit -q

from pathlib import Path
import json
import zipfile
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import warnings
from sklearn.metrics import roc_auc_score
from pytabkit import RealMLP_TD_Classifier
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

# ---- Config ----
COMP_SLUG = "playground-series-s6e2"
KAGGLE_COMP_DIR = Path("/kaggle/input/competitions/playground-series-s6e2")
KAGGLE_EXT_PATH = Path("/kaggle/input/datasets/neurocipher/heartdisease/Heart_Disease_Prediction.csv")

LOCAL_DATA_DIR = Path("data/raw")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

NEED_FILES = ["train.csv", "test.csv", "sample_submission.csv"]

In [ ]:
def run(cmd: list[str]) -> None:
    p = subprocess.run(cmd, capture_output=True, text=True)
    p.check_returncode()


def ensure_kaggle_cli() -> None:
    try:
        pass
    except Exception:
        subprocess.check_call(["pip", "-q", "install", "kaggle"])


def ensure_kaggle_json_interactive_colab(dst: Path = Path("/content/kaggle.json")) -> Path:
    """
    In Colab: open upload dialog if /content/kaggle.json is missing.
    In non-Colab: just require the file to exist.
    """
    if dst.exists():
        print("Found:", dst)
        return dst

    try:
        from google.colab import files  # type: ignore
    except Exception:
        raise FileNotFoundError(
            f"{dst} not found. Please place kaggle.json at {dst} (Colab) "
            "or provide credentials another way."
        )

    print("Upload your kaggle.json (Kaggle -> Account -> API -> Create New Token)")
    uploaded = files.upload()
    cand = None
    if "kaggle.json" in uploaded:
        cand = "kaggle.json"
    else:
        for name in uploaded.keys():
            if name.endswith(".json"):
                cand = name
                break
    if cand is None:
        raise FileNotFoundError("Upload failed: no .json file received.")

    Path(cand).rename(dst)
    print("Saved to:", dst)
    return dst


def install_kaggle_json(src: Path) -> None:
    """
    Copy /content/kaggle.json -> ~/.kaggle/kaggle.json (chmod 600)
    """
    if not src.exists():
        raise FileNotFoundError(f"{src} not found.")

    dst_dir = Path.home() / ".kaggle"
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / "kaggle.json"

    dst.write_bytes(src.read_bytes())
    try:
        dst.chmod(0o600)
    except Exception:
        pass

    cfg = json.loads(dst.read_text())
    if "username" not in cfg or "key" not in cfg:
        raise ValueError("kaggle.json is missing 'username' or 'key'.")
    print(f"Installed kaggle.json for user: {cfg['username']}")


def local_data_ready(data_dir: Path) -> bool:
    return all((data_dir / f).exists() for f in NEED_FILES)


def download_competition_to(data_dir: Path) -> None:
    """
    Download competition zip(s) and extract into data_dir.
    Assumes kaggle CLI + credentials are ready.
    """
    run(["kaggle", "config", "view"])
    run(["kaggle", "competitions", "download", "-c", COMP_SLUG, "-p", str(data_dir), "--force"])

    zips = list(data_dir.glob("*.zip"))
    if not zips:
        raise FileNotFoundError(f"No zip found in {data_dir} after download.")

    for zp in zips:
        with zipfile.ZipFile(zp, "r") as zf:
            zf.extractall(data_dir)
        print("Unzipped:", zp.name)

    if not local_data_ready(data_dir):
        missing = [f for f in NEED_FILES if not (data_dir / f).exists()]
        raise FileNotFoundError(f"Download/unzip finished but missing: {missing}")


In [ ]:
if KAGGLE_COMP_DIR.exists():
    DATA_SRC = "kaggle"
    data_dir = KAGGLE_COMP_DIR
    print("Using Kaggle mounted competition data:", data_dir)
else:
    DATA_SRC = "local"
    data_dir = LOCAL_DATA_DIR
    if local_data_ready(data_dir):
        print("Using local data (already present):", data_dir)
    else:
        print("Local data missing -> download using kaggle.json")
        ensure_kaggle_cli()
        kaggle_json_src = ensure_kaggle_json_interactive_colab(Path("/content/kaggle.json"))
        install_kaggle_json(kaggle_json_src)
        download_competition_to(data_dir)
        print("Download complete -> using local data:", data_dir)


# ---- Load ----
train = pd.read_csv(data_dir / "train.csv")
test  = pd.read_csv(data_dir / "test.csv")
sub   = pd.read_csv(data_dir / "sample_submission.csv")

# external dataset: only available if mounted on Kaggle; optional
original = pd.read_csv(KAGGLE_EXT_PATH) if KAGGLE_EXT_PATH.exists() else None

print("train:", train.shape, "test:", test.shape, "sub:", sub.shape, "original:", None if original is None else original.shape)
print("DATA_SRC:", DATA_SRC)


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_STATE = 42
N_FOLDS = 5
USE_ALL_CAT = True

print(f"Using device: {DEVICE}")

### Data download

In [ ]:
display(train.head())
display(test.head())
display(sub.head())

In [ ]:
# Shapes
print("train:", train.shape)
print("test:", test.shape)

# Column diffs
train_cols = set(train.columns)
test_cols = set(test.columns)
print("Only in train:", sorted(train_cols - test_cols))
print("Only in test:", sorted(test_cols - train_cols))

# dtypes
train.dtypes.to_frame("dtype").head(30)


### Data Preprocessing

## 1) What the feature space looks like

Across the notes, the canonical raw feature set is **13 predictors + id + target**.

**Target**

- `Heart Disease` (mapped `Absence` → 0, `Presence` → 1)

**Numerical (5)**

- `Age`, `BP`, `Cholesterol`, `Max HR`, `ST depression`

**Categorical (8)**

- `Sex`, `Chest pain type`, `FBS over 120`, `EKG results`,
  `Exercise angina`, `Slope of ST`, `Number of vessels fluro`, `Thallium`

Many EDA notebooks treat everything as numeric after encoding, even though those 8 are semantically categorical.


In [ ]:
TARGET_COL = "Heart Disease"
ID_COL = "id"

NUM_COLS = ["Age", "BP", "Cholesterol", "Max HR", "ST depression"]
CAT_COLS = [
    "Sex",
    "Chest pain type",
    "FBS over 120",
    "EKG results",
    "Exercise angina",
    "Slope of ST",
    "Number of vessels fluro",
    "Thallium",
]
FEATURE_COLS = NUM_COLS + CAT_COLS


def encode_target(series: pd.Series) -> pd.Series:
    if series.dtype == "O":
        return series.map({"Absence": 0, "Presence": 1})
    return series


y = encode_target(train[TARGET_COL])

print("Target:", TARGET_COL)
print("Numerical:", NUM_COLS)
print("Categorical:", CAT_COLS)

missing_cols = [c for c in [ID_COL, TARGET_COL] + FEATURE_COLS if c not in train.columns]
print("Missing columns in train:", missing_cols)


## 2) EDA conclusions you should internalize

### Data quality


In [ ]:
train_missing = train.isna().sum()
test_missing = test.isna().sum()
print("Train missing total:", int(train_missing.sum()))
print("Test missing total:", int(test_missing.sum()))

display(train_missing[train_missing > 0])
display(test_missing[test_missing > 0])

print("Train duplicates:", int(train.duplicated().sum()))
print("Test duplicates:", int(test.duplicated().sum()))


### Target balance


In [ ]:
counts = y.value_counts()
print(counts)
print(counts / counts.sum())

ratio = counts.max() / counts.min()
print("Class balance ratio (max/min):", ratio)


### Train vs Test shift


In [ ]:
from scipy.stats import ks_2samp

ks_rows = []
for col in FEATURE_COLS:
    stat, p = ks_2samp(train[col], test[col])
    ks_rows.append({"feature": col, "statistic": stat, "p_value": p, "aligned": p > 0.05})

ks_df = pd.DataFrame(ks_rows).sort_values("p_value")
print("Aligned features:", ks_df["aligned"].sum(), "/", len(ks_df))
display(ks_df)


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

adv_df = pd.concat([train[FEATURE_COLS], test[FEATURE_COLS]], axis=0).reset_index(drop=True)
adv_label = np.r_[np.zeros(len(train)), np.ones(len(test))]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aucs = []
for tr_idx, va_idx in skf.split(adv_df, adv_label):
    X_tr, X_va = adv_df.iloc[tr_idx], adv_df.iloc[va_idx]
    y_tr, y_va = adv_label[tr_idx], adv_label[va_idx]
    model = LogisticRegression(max_iter=1000)
    model.fit(X_tr, y_tr)
    preds = model.predict_proba(X_va)[:, 1]
    aucs.append(roc_auc_score(y_va, preds))

print("Adversarial validation AUC (mean):", float(np.mean(aucs)))


### Skew + outliers


In [ ]:
skew_vals = train[FEATURE_COLS].skew()

outlier_counts = {}
for col in FEATURE_COLS:
    q1 = train[col].quantile(0.25)
    q3 = train[col].quantile(0.75)
    iqr = q3 - q1
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    outlier_counts[col] = int(((train[col] < lo) | (train[col] > hi)).sum())

summary = pd.DataFrame({
    "skew": skew_vals,
    "outlier_count": pd.Series(outlier_counts)
}).sort_values("skew", key=lambda s: s.abs(), ascending=False)

print("Highly skewed (|skew| > 1):", int((summary["skew"].abs() > 1).sum()))
print("Features with outliers:", int((summary["outlier_count"] > 0).sum()))

display(summary)


### Strongest signals (feature–target)


In [ ]:
df_corr = train[FEATURE_COLS].copy()
df_corr["target"] = y
corrs = df_corr.corr(numeric_only=True)["target"].drop("target").sort_values(ascending=False)

print("Top 5 correlations with target:")
display(corrs.head(5))

print("Top 3:")
print(corrs.head(3))


### Multicollinearity


In [ ]:
corr = train[FEATURE_COLS].corr(numeric_only=True)
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "corr"})
)
strong_pairs = pairs[pairs["corr"].abs() > 0.7].sort_values("corr", key=lambda s: s.abs(), ascending=False)

if strong_pairs.empty:
    print("No |r| > 0.7 pairs found")
else:
    display(strong_pairs)


## 3) Feature engineering people actually used in the notes


### A) External target stats from original dataset


In [ ]:
if original is None:
    print("No external dataset mounted; skipping external target stats.")
else:
    original = original.copy()
    if TARGET_COL in original.columns:
        original[TARGET_COL] = encode_target(original[TARGET_COL])

    train_ext = train.copy()
    test_ext = test.copy()

    global_stats = {
        "mean": float(original[TARGET_COL].mean()),
        "median": float(original[TARGET_COL].median()),
        "std": float(original[TARGET_COL].std()),
        "skew": float(original[TARGET_COL].skew()),
        "count": float(original[TARGET_COL].count()),
    }

    for col in FEATURE_COLS:
        stats = original.groupby(col)[TARGET_COL].agg(["mean", "median", "std", "skew", "count"]).reset_index()
        stats = stats.rename(columns={
            "mean": f"orig_{col}_mean",
            "median": f"orig_{col}_median",
            "std": f"orig_{col}_std",
            "skew": f"orig_{col}_skew",
            "count": f"orig_{col}_count",
        })
        train_ext = train_ext.merge(stats, on=col, how="left")
        test_ext = test_ext.merge(stats, on=col, how="left")

        for suffix, val in global_stats.items():
            cname = f"orig_{col}_{suffix}"
            train_ext[cname] = train_ext[cname].fillna(val)
            test_ext[cname] = test_ext[cname].fillna(val)

    print("train_ext shape:", train_ext.shape)
    print("test_ext shape:", test_ext.shape)
    display(train_ext.filter(regex="^orig_").head())


### B) Frequency encoding


In [ ]:
all_df = pd.concat([train[FEATURE_COLS], test[FEATURE_COLS]], axis=0)
train_freq = train.copy()
test_freq = test.copy()

for col in FEATURE_COLS:
    freq = all_df[col].value_counts()
    train_freq[f"{col}_freq"] = train[col].map(freq)
    test_freq[f"{col}_freq"] = test[col].map(freq)

print("train_freq shape:", train_freq.shape)
print("test_freq shape:", test_freq.shape)

display(train_freq[[f"{c}_freq" for c in FEATURE_COLS]].head())


### C) Binning (KBinsDiscretizer)


In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

kb = KBinsDiscretizer(n_bins=10, strategy="uniform", encode="ordinal")
train_bins = train.copy()
test_bins = test.copy()

train_b = kb.fit_transform(train[NUM_COLS])
test_b = kb.transform(test[NUM_COLS])

for i, col in enumerate(NUM_COLS):
    train_bins[f"{col}_bin"] = train_b[:, i]
    test_bins[f"{col}_bin"] = test_b[:, i]

sample_col = f"{NUM_COLS[0]}_bin"
print("Sample bin counts for", sample_col)
print(train_bins[sample_col].value_counts().sort_index())


### D) Target mean/count encodings inside a pipeline


In [ ]:
def oof_target_stats(train_df, test_df, col, y, n_splits=5, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof_mean = pd.Series(index=train_df.index, dtype=float)
    oof_count = pd.Series(index=train_df.index, dtype=float)

    for tr_idx, va_idx in skf.split(train_df, y):
        tr = train_df.iloc[tr_idx]
        y_tr = y.iloc[tr_idx]
        stats = pd.DataFrame({col: tr[col], "target": y_tr}).groupby(col)["target"].agg(["mean", "count"])
        oof_mean.iloc[va_idx] = train_df.iloc[va_idx][col].map(stats["mean"])
        oof_count.iloc[va_idx] = train_df.iloc[va_idx][col].map(stats["count"])

    full_stats = pd.DataFrame({col: train_df[col], "target": y}).groupby(col)["target"].agg(["mean", "count"])
    test_mean = test_df[col].map(full_stats["mean"]).fillna(y.mean())
    test_count = test_df[col].map(full_stats["count"]).fillna(0)

    oof_mean = oof_mean.fillna(y.mean())
    oof_count = oof_count.fillna(0)

    return oof_mean, oof_count, test_mean, test_count


train_te = train.copy()
test_te = test.copy()

for col in CAT_COLS:
    oof_mean, oof_count, test_mean, test_count = oof_target_stats(train, test, col, y)
    train_te[f"{col}_te_mean"] = oof_mean
    train_te[f"{col}_te_count"] = oof_count
    test_te[f"{col}_te_mean"] = test_mean
    test_te[f"{col}_te_count"] = test_count

print("train_te shape:", train_te.shape)
print("test_te shape:", test_te.shape)

display(train_te.filter(regex="_te_(mean|count)$").head())


### E) Robust scaling + log transforms


In [ ]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
train_scaled = train.copy()
test_scaled = test.copy()

train_scaled[NUM_COLS] = scaler.fit_transform(train[NUM_COLS])
test_scaled[NUM_COLS] = scaler.transform(test[NUM_COLS])

train_log = train.copy()
test_log = test.copy()
for col in NUM_COLS:
    train_log[f"{col}_log"] = np.log1p(train[col])
    test_log[f"{col}_log"] = np.log1p(test[col])

print("Skew before:")
print(train[NUM_COLS].skew())
print("Skew after log1p:")
print(train_log[[f"{c}_log" for c in NUM_COLS]].skew())
